# 3. Training Data

แบ่งข้อมูลเป็น **train / validation / test = 70% / 15% / 15%** ก่อน fit preprocessing ใด ๆ เพื่อป้องกัน data leakage

In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [9]:
df = pd.read_parquet("../data/processed/student_data_pre_impute.parquet")
df.head()

,Age,Year_of_Study,Department,Residence_Type,Attendance_Percent,Study_Hours_Per_Day,Previous_GPA,Backlogs,Screen_Time_Hours,Part_Time_Job,Family_Income_Bracket,Financial_Stress_Score,Family_Support_Score,Stress_Level,Anxiety_Score,Motivation_Score,Counseling_Access,Dropout_Risk
0,23,2,Science,Hostel,93.0,4.1,7.13,2.0,4.9,Yes,Upper-Middle,5.5,10.0,4.0,NaN,7.0,No,No
1,20,4,Engineering,Day Scholar,94.3,0.0,10.00,0.0,NaN,Yes,Low,3.7,4.0,5.5,5.1,9.3,Yes,Yes
2,24,2,Business,PG/Rented,61.2,2.6,8.02,0.0,3.8,No,Middle,4.8,8.1,9.0,5.2,5.7,No,Yes
3,21,4,Medicine,Hostel,66.9,3.1,5.34,1.0,5.9,Yes,Lower-Middle,7.0,6.6,2.8,4.2,5.1,No,Yes
4,23,1,Engineering,None,84.8,2.9,4.45,1.0,5.0,No,Lower-Middle,1.3,7.3,0.9,6.4,8.1,No,No


In [10]:
X = df.drop(columns="Dropout_Risk")
y = df["Dropout_Risk"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

In [11]:
print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
print("X_test :", X_test.shape)

X_train: (560, 17)
X_valid: (120, 17)
X_test : (120, 17)


In [12]:
for name, target in [("train", y_train), ("valid", y_valid), ("test", y_test)]:
    print(f"\n{name}")
    print(target.value_counts(normalize=True).round(4))


train
Dropout_Risk
Yes    0.5464
No     0.4536
Name: proportion, dtype: float64

valid
Dropout_Risk
Yes    0.55
No     0.45
Name: proportion, dtype: float64

test
Dropout_Risk
Yes    0.5417
No     0.4583
Name: proportion, dtype: float64


Preprocessing จะ **fit จาก training data เท่านั้น** ภายใน `Pipeline` ในขั้น model development / hyperparameter tuning ส่วน validation และ test จะถูก transform ด้วยค่าที่เรียนรู้จาก training data เท่านั้น

In [13]:
train_df = pd.concat([X_train, y_train], axis=1)
valid_df = pd.concat([X_valid, y_valid], axis=1)
test_df = pd.concat([X_test, y_test], axis=1)

In [14]:
from pathlib import Path

Path("../data/processed").mkdir(parents=True, exist_ok=True)
train_df.to_parquet("../data/processed/train_pre_impute.parquet", index=False)
valid_df.to_parquet("../data/processed/valid_pre_impute.parquet", index=False)
test_df.to_parquet("../data/processed/test_pre_impute.parquet", index=False)